In [ ]:
import pandas as pd
import re
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import umap
from sklearn.datasets import load_digits


import hdbscan
import sklearn.cluster as cluster
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score


In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
df_col

In [ ]:
df0[["q45_clé", "q24_research_fields"]].loc[df0.q24_research_fields.str.contains("LS6 Immunité, infection et immunothérapie")]

# Profil disciplinaire

In [ ]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df0.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data, df_explode

In [ ]:
df0["q24_research_fields"]= df0.apply(lambda row: row.q24_research_fields.replace(";","|") ,1 )


In [ ]:
gb_data, df_exp = split_multiple_choices(df0, column='q24_research_fields', index = "q45_clé")


In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(1, figsize=(10, 20))

# Plot the total crashes
sns.set_color_codes("pastel")

for n, col in enumerate(['q24_research_fields']):
    gb_data, df_rf = split_multiple_choices(df0, column=col, index = "q45_clé")
    sns.barplot(x="total_freq", y=col, data=gb_data,
                label="Non", color="b", ax=ax)
    sns.barplot(x="freq", y=col, data=gb_data,
                label="Oui", color="r", ax=ax)
    titre = df_col.question.loc[df_col.label==col].iloc[0]
    ax.yaxis.set_label_text("")
    ax.xaxis.set_label_text("")
    ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
#plt.savefig(f"viz/multiple_choice_1.png", bbox_inches='tight', dpi = 200)

In [ ]:
df_rf = df_exp[["q45_clé", 'q24_research_fields']].copy()
df_rf.loc[df_rf.q24_research_fields.str.contains("SH"), "q24_research_domain"] = "Sciences sociales et humaines"
df_rf.loc[df_rf.q24_research_fields.str.contains("LS"), "q24_research_domain"] = "Sciences de la vie"
df_rf.loc[df_rf.q24_research_fields.str.contains("PE"), "q24_research_domain"] = "Sciences physiques et ingénierie"
df_rf["value"] = 1
df_rf.q24_research_fields.value_counts()
df_rf

## Reduce dimension for research fields
df_rf1

# Clusterisation iindividus

## Unsupervised dimension reduction

In [ ]:
affil = pd.read_csv("list_affiliation.csv", sep =",")
affil.loc[affil.q45_clé.str.contains("2Y")]

## Semisupervised dimension reduction

In [ ]:
#### df_rf5.to_csv("embedding_indiv_research_fields.csv", sep = ",", index = False)

dfn = pd.read_csv("buparis8_chercheurs_besoins_accompagnement_5-11-2026_16_7.csv", sep =";")
dfn = dfn.rename(columns={"142. Nom :":"q42_nom", "143. Prénom :":"q42_prenom", "146. Clé":"q45_clé"})

df_rf5 = pd.read_csv("embedding_indiv_research_fields.csv", sep =",")


affil1 = affil.merge(dfn[["q45_clé","q42_nom","q42_prenom"]], on = ["q45_clé"], how = "left")
affil1

In [ ]:
with open("tableau_cluster_ind.txt", "r", newline='') as fin:
    lines = fin.readlines()

dict_cluster = {}
list_cluster = []
for l in lines :
    l_split = l.split(",")
    if len(l_split) > 1:
        dict_cluster[l_split[0]] = l_split[-1].strip()
        if l_split[-1].strip() not in list_cluster:
            list_cluster.append(l_split[-1].strip())
    else:
        pass

no_cluster = {}
for c in list_cluster:
    no_cluster[c] = list_cluster.index(c)
no_cluster


In [ ]:
df_rf6 = df_rf5.drop(columns=['cluster',
       'lab_cluster', 'new_clust', 'semi_cluster', 'semi_lab_cluster']).loc[~df_rf5.q45_clé.isin(["9EAP-NB4B","QNYZ-3MH2","H5X6-KL2Z",'PS8W-TJ9P'])]
df_rf6["cluster"] = df_rf6.q45_clé.map(dict_cluster)
df_rf6["no_cluster"] = df_rf6.cluster.map(no_cluster)

In [ ]:
df_rf6.columns[1:22]

In [ ]:
fig, ax = plt.subplots(1, figsize=(10,10))

matrix_ind = df_rf6[df_rf6.columns[1:22]].values
matrix_ind


embedding = umap.UMAP(n_neighbors=5,
                      min_dist=0.1,
                      n_components=2,
                      metric='cosine', random_state =42).fit_transform(matrix_ind, target = df_rf6[df_rf6.columns[-1]].values)

sns.scatterplot(x=embedding[:,0], y=embedding[:,1],  ax=ax, hue = [str(x) for x in df_rf6.cluster])


In [ ]:
dict_nom = dict(zip(affil1.q45_clé, affil1.q42_nom))
dict_prenom = dict(zip(affil1.q45_clé, affil1.q42_prenom))

list_row = []
for n, x in enumerate(embedding):
    id_key = df_rf6[df_rf6.columns[0]].values[n]
    try:
        nom = dict_nom[id_key].lower()
    except:
        nom = "not defined"
    try:
        prenom = dict_prenom[id_key].lower()
    except:
        prenom = "not defined"
    dict_row = {"q45_clé": id_key,
                      "x": x[0],
                      "y": x[1],
                      "nom": nom,
                      "prenom": prenom
                     }
    list_row.append(dict_row)
    

df_emb = pd.DataFrame.from_dict(list_row)
df_emb

In [ ]:
import plotly.express as px


fig_2d = px.scatter(
    df_emb, x="x", y="y",
    color=df_rf6.cluster, hover_data=['q45_clé', 'nom', 'prenom'],
)

fig_2d.write_html('plotly_embedding.html')

In [ ]:
labels = hdbscan.HDBSCAN(
    min_samples=4,
    min_cluster_size=4, gen_min_span_tree=True
).fit(embedding)

labels.single_linkage_tree_.plot()
print(len(set(labels.labels_)))
print(len([x for x in labels.labels_ if x == -1]))

In [ ]:
labels.minimum_spanning_tree_.plot(edge_cmap='viridis', 
                                      edge_alpha=0.6, 
                                      node_size=60, 
                                      edge_linewidth=2)


In [ ]:
name_column = df_rf6.columns[1:22]
name_column

for x in name_column:
    abrev = re.search(r"\w*\d+", x)
    print(abrev.group())

[re.search(r"\w*\d+", x).group() for x in name_column ]

In [ ]:
#%%capture cap
dict_clusters = {}

for n, x in enumerate(df_rf6.q45_clé):
    dict_clusters[x] = labels.labels_[n]


with open("tableau_cluster.txt", 'w') as fout: 
    for x in range(-1,len(set(labels.labels_))):
        fout.write(f"Cluster : {x}\n\n")
        name_column = df_rf6.columns[1:22]
        abrev = ",".join([re.search(r"\w*\d+", x).group() for x in name_column ])
        fout.write(f"colmuns : {abrev}\n")
        compteur = 0
        for v in dict_clusters:
            if dict_clusters[v] == x:
                compteur+=1
                dtmp0 = df_rf6.merge(affil, on = "q45_clé", how = "left")
                dtmp = dtmp0[dtmp0.columns[35]].loc[dtmp0.q45_clé == v].values
                dtmp1 = dtmp0[dtmp0.columns[1:22]].loc[dtmp0.q45_clé == v].values
                fout.write(f'{v}: {",".join([str(x) for x in dtmp1[0]])} | {dtmp[0]} \n')
        fout.write("\n=====================\n")


In [ ]:
with open("tableau_cluster.txt", "r", newline='') as fin:
    lines = fin.readlines()

row_cluster = []
no_cluster = {}
for l in lines :
    if re.match("Cluster", l):
        #print(l)
        id_area = re.findall(r"\d+", l)
        area = l.split(":")[-1].strip()
        no_cluster[id_cluster[0]] = area
    else:
        if re.search("-",l):
            l_split = l.split(":")
            l_cluster = l_split[-1].split("|")
            #print(l_split[0], list_cluster)
            dict_row = {'q45_clé': l_split[0].strip(),
                       'no_area': id_area[0],
                       'area': area,
                       'fields': l_cluster[-1].strip()}
            row_cluster.append(dict_row)
         

df_cluster = pd.DataFrame.from_dict(row_cluster)

In [ ]:
df_emb1  = df_emb.merge(df_cluster, on = ["q45_clé"], how ="left")
list_point = [x for x in df_emb.q45_clé.loc[df_emb.new_cluster==-1]]
df_rf6[df_rf6.columns[0:23]].loc[df_rf6.q45_clé.isin(list_point)]

In [ ]:

fig_2d = px.scatter(
    df_emb1, x="x", y="y",
    color="area", hover_data=['q45_clé', 'nom', 'prenom'],
)

fig_2d.write_html('plotly_embedding2.html')